# Gravando o resultado: overwrite, append e Delta Lake

## Demanda da aula 5

O time de plataforma da Arena Analytics chegou com o último pedido do projeto: **colocar o pipeline em produção**.

Até aqui, os resultados foram calculados em notebooks, e só os passes da aula 4 foram gravados. Em produção, o pipeline
precisa rodar sozinho, chamado por um agendador, e **gravar** os resultados num lugar em que outros sistemas (o painel
da transmissão, o time de analytics) consigam ler. E precisa gravar de forma **segura**: sem perder dado e sem duplicar
dado, mesmo se rodar duas vezes.

E o primeiro teste já tem data: o fornecedor mandou os dados da **Copa de 2026**, que precisam entrar no histórico.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Definir como e em que formato o pipeline grava | `01_overwrite_e_delta` |
| Task 2 | Organizar a saída por Copa | `02_partitionby` |
| Task 3 | Controlar os arquivos gravados e carregar uma Copa nova sem apagar o histórico | `03_repartition_e_overwrite_dinamico` |
| Task 4 | Colocar o pipeline em produção, rodando fora do notebook, e carregar a Copa de 2026 | `04_locais_e_pipeline_final` |

**Pronto quando:** o pipeline roda com um comando, fora do notebook; grava em Delta Lake, organizado por Copa, com um
arquivo por pasta; carrega a Copa de 2026 sem apagar nem duplicar o histórico, mesmo se rodar duas vezes; e a saída é
conferida automaticamente.

## Task 1: definir como e em que formato o pipeline grava

A sessão já vem pronta, com três linhas a mais: elas carregam o **Delta Lake**, que você conhece neste vídeo.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-escrita")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"

spark

## As partidas do projeto

`ler_partidas(pasta)`: a mesma leitura serve para o histórico e para a Copa de 2026, que chega em outra pasta.

A próxima célula já vem pronta: ela só apaga as pastas que este notebook grava, para ele poder rodar de novo.

In [ ]:
import shutil
for pasta in ["partidas_parquet", "partidas_delta"]:
    shutil.rmtree(f"{OUT}/{pasta}", ignore_errors=True)

## E se o job rodar de novo?

| Modo | Se o destino já existe |
|---|---|
| `append` | acrescenta arquivos novos |
| `overwrite` | apaga e grava de novo |

Sem modo nenhum, o padrão é dar **erro** se a pasta já existe: o projeto sempre declara o modo.

**`append` num job que pode rodar de novo duplica o dado, sem erro.** O modo do projeto é o `overwrite`.

## O risco do overwrite

E se o job rodar com um filtro de teste esquecido?

O histórico antigo sumiu, e o Parquet não tem "desfazer": no `overwrite`, o dado é apagado antes de o novo ser
gravado, e a escrita não é atômica.

## Os formatos e o Delta Lake

| Formato | Como guarda |
|---|---|
| **CSV** | texto, uma linha por registro, sem tipos |
| **JSON** | texto, repete o nome de cada coluna em cada linha |
| **Parquet** | binário e colunar, com o schema (vídeo 4.4) |

O **Delta Lake** guarda o dado em **Parquet** e acrescenta a pasta `_delta_log/`, com um registro de cada escrita:

- **escrita atômica**: só vale quando é registrada no log, no fim;
- **versões**: as anteriores continuam disponíveis.

Os `part-...parquet` estão lá; a novidade é o `_delta_log`, com o registro da **versão 0**.

## O mesmo engano, agora em Delta

A versão atual ficou com o engano, mas os arquivos da versão 0 continuam lá:

Para desfazer, leia a versão 0 e grave de novo:

## A decisão do projeto

| Formato | Use para |
|---|---|
| **CSV** | trocar arquivos com pessoas e sistemas antigos |
| **JSON** | dado semiestruturado, respostas de APIs |
| **Parquet** | dado analítico gravado de uma vez |
| **Delta Lake** | tabelas de pipeline, regravadas, com transação e histórico |

**Decisão:** `overwrite`, em **Delta Lake**.

Dois problemas para os próximos vídeos: o painel pergunta por **uma Copa** e o Spark lê a tabela inteira; e o
`overwrite` troca **a tabela inteira**, então a Copa de 2026 apagaria as outras da versão atual.

## Resultado da task 1

`overwrite` em Delta Lake. Próxima task: organizar a saída por Copa.

## Para consultar depois

- **Os outros dois modos:** `error` (ou `errorifexists`), o padrão, lança erro se o destino existe; `ignore` não faz
  nada se o destino existe, sem avisar.
- **A mesma escrita, de outro jeito:** `df.write.format("parquet").mode("overwrite").save(caminho)` é o mesmo que
  `df.write.mode("overwrite").parquet(caminho)`.
- **`MERGE`:** quando o lote que chega é **parcial**, como as partidas do dia durante uma Copa, o Delta permite
  atualizar as linhas que já existem e inserir as novas numa operação só, com `MERGE INTO`. No projeto, cada lote é uma
  Copa inteira, e o vídeo 5.3 mostra o caminho mais simples para esse caso.
- **`VACUUM`:** apaga os arquivos que não fazem parte de nenhuma versão recente. Sem ele, as versões antigas ocupam
  espaço para sempre.
- **Outros formatos de tabela:** Apache Iceberg e Apache Hudi seguem a mesma ideia do Delta, com metadados sobre
  arquivos Parquet.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()